# 03 — HPO Preprocessing

Parses `phenotype.hpoa` (34 MB), filters to ID-relevant diseases, constructs frequency-weighted HPO term feature matrix (Aspect==P, no NOT qualifiers), adds gene count features, fits StandardScaler, and saves all arrays to `datasets/processed/facial/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/earlyMind'):
    !git clone https://github.com/Rickykapoor/earlyMind.git /content/earlyMind
%cd /content/earlyMind
!git pull origin main

In [ ]:
!pip install -q mne nibabel nilearn openneuro-py torch torchvision \
  streamlit plotly scikit-learn pytorch-tabnet \
  xgboost catboost tqdm pyyaml

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# CELL 4: Data Loader (Supports DVC on Mounted Google Drive, Drag-and-Drop, or Auto Download)
import os, glob, shutil
os.makedirs('datasets/facial/hpo', exist_ok=True)

# OPTION A: If your DVC cache is in your own Google Drive, point DVC to your mounted drive!
# => Update this to the exact folder path of your DVC storage inside your Google Drive
MY_DRIVE_DVC_PATH = '/content/drive/MyDrive/earlyMind_DVC' # <-- Update this!

if os.path.exists(MY_DRIVE_DVC_PATH):
    print(f'Found local DVC remote at {MY_DRIVE_DVC_PATH}...')
    os.system(f'dvc remote add -d local_gdrive {MY_DRIVE_DVC_PATH} --force')
    os.system('dvc pull')
    print('DVC pull complete from mounted Google Drive!')

# OPTION B: Check if user dragged-and-dropped the raw files directly to /content/
if len(glob.glob('/content/*.hpoa')) > 0:
    print('Found raw files in /content/! Moving them to datasets/facial/hpo...')
    os.system(f'mv /content/*.hpoa datasets/facial/hpo/')
    os.system(f'mv /content/*.hpoa datasets/facial/hpo/ 2>/dev/null')

# 2. Check if user dragged-and-dropped the .tar.gz file to /content/
elif os.path.exists('/content/facial_raw.tar.gz'):
    print('Found facial_raw.tar.gz in /content/! Extracting...')
    os.system(f'tar -xzf /content/facial_raw.tar.gz -C datasets/facial')

# 3. Fallback: Automatically download from GitHub Releases
else:
    print('Downloading dataset automatically from GitHub Releases...')
    os.system(f'wget -qO facial_raw.tar.gz https://github.com/Rickykapoor/earlyMind/releases/download/v1.0.0-data/facial_raw.tar.gz')
    os.system(f'tar -xzf facial_raw.tar.gz -C datasets/facial')

print('Datasets ready.')


In [ ]:
import sys
sys.path.insert(0, '/content/earlyMind')

from src.config import cfg
import pandas as pd

print('HPO raw dir: ', cfg.paths.hpo_raw)

# Inspect phenotype.hpoa header
hpoa_path = cfg.paths.hpo_raw / 'phenotype.hpoa'
print('\nFirst 5 lines of phenotype.hpoa:')
with open(hpoa_path) as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i >= 5:
            break

In [ ]:
# Quick stats on raw hpoa file
df_hpo_raw = pd.read_csv(hpoa_path, sep='\t')
df_hpo_raw.columns = [c.lstrip('#').strip() for c in df_hpo_raw.columns]
print('Shape:', df_hpo_raw.shape)
print('Columns:', df_hpo_raw.columns.tolist())
print(f'Unique diseases: {df_hpo_raw.iloc[:, 1].nunique()}')
display(df_hpo_raw.head())

In [ ]:
# Run full HPO preprocessing
from src.data.hpo_loader import process_hpo_dataset

results = process_hpo_dataset(
    hpo_dir=cfg.paths.hpo_raw,
    output_dir=cfg.paths.hpo_processed,
)

X = results['X']
y = results['y']
dq = results['dq']

print(f'\nMatrix shape: {X.shape}')
print(f'Labels: {int((y==1).sum())} ID-positive, {int((y==0).sum())} non-ID')
print(f'DQ range: [{dq.min():.1f}, {dq.max():.1f}]')

In [ ]:
# Plot top disease categories
import matplotlib.pyplot as plt
import numpy as np

disease_names = results['disease_names']
feat_names    = results['feat_names']

# Column sum = term frequency
term_freq = (X > 0).sum(axis=0)
top_idx   = np.argsort(term_freq)[::-1][:20]
top_terms = [feat_names[i] for i in top_idx]
top_freq  = term_freq[top_idx]

plt.figure(figsize=(14, 5))
plt.bar(range(20), top_freq)
plt.xticks(range(20), top_terms, rotation=60, ha='right', fontsize=8)
plt.ylabel('# Diseases with term')
plt.title('Top 20 Most Common HPO Terms in ID Diseases')
plt.tight_layout()
plt.savefig(str(cfg.paths.reports) + '/hpo_top_terms.png', dpi=100)
plt.show()

print(f'Feature matrix: {X.shape[0]} diseases × {X.shape[1]} HPO features')

In [ ]:
# Reload updated config to confirm hpo_n_features was updated
import importlib, src.config
importlib.reload(src.config)
from src.config import cfg as cfg_new
print(f'params.yaml hpo_n_features updated to: {cfg_new.model.hpo_n_features}')

In [ ]:
!git add checkpoints/ reports/ datasets/processed/ params.yaml
!git commit -m 'colab: 03_hpo_preprocess completed'
!git push origin main